# **Project: Nghiên cứu mô hình học đa phương thức dựa trên đồ thị, chuẩn đoán bệnh nghề nghiệp dựa trên ảnh X-quang và dữ liệu lâm sàng**

# Table of Contents
* [1. Loading data](#loading-data)
* [2. Feature Engineering and Data Wrangling](#fe-dw)

In [ ]:
from google.colab import drive
# Mount Google Drive vào thư mục /content/drive
drive.mount('/content/drive')

: 

In [ ]:
# === 1) (Colab) Tải dữ liệu đúng 1 lần + lọc cột (chưa lưu Drive) ===

import re
from io import BytesIO

import pandas as pd
import requests

# Google Sheet dữ liệu gốc
RAW_SHEET_URL = "https://docs.google.com/spreadsheets/d/1MBTFK4dw-PBbh8NA2HEaNo0_CVemTJfF/edit?usp=drive_link&ouid=103230800443557501737&rtpof=true&sd=true"

# --- Download Google Sheet as .xlsx (1 lần) ---
m = re.search(r"/spreadsheets/d/([^/]+)/", RAW_SHEET_URL)
if not m:
    raise ValueError("Không trích được spreadsheet id từ RAW_SHEET_URL. Hãy kiểm tra lại link.")

raw_sid = m.group(1)
raw_export_url = f"https://docs.google.com/spreadsheets/d/{raw_sid}/export?format=xlsx"

resp = requests.get(raw_export_url)
if resp.status_code != 200 or (resp.headers.get("content-type", "").find("spreadsheet") == -1 and resp.content[:4] != b"PK\x03\x04"):
    raise RuntimeError(
        "Không tải được file .xlsx từ Google Sheet.\n"
        "- Hãy đảm bảo file đang bật quyền 'Anyone with the link can view' hoặc bạn đã đăng nhập đúng tài khoản trên Colab.\n"
        f"- status_code={resp.status_code}, content-type={resp.headers.get('content-type')}"
    )

# Load trực tiếp từ bytes (không lưu file, tránh tải lại nhiều lần)
df_raw = pd.read_excel(BytesIO(resp.content))

print("✅ Loaded raw data")
print(f"- Source export: {raw_export_url}")
print(f"- Shape: {df_raw.shape}")
print("- First 10 columns:", list(df_raw.columns)[:10])

# --- Lọc cột ---
# 1) Nhóm thông tin định danh / nhãn chi tiết không dùng cho mô hình
# 2) Nhóm kết quả đọc phim X-quang (tránh rò rỉ sang nhánh timeseries)
explicit_drop = {
    "id",
    "tinh",
    "hoten",
    "sdt",
    "sobh",
    "bnncuthe",
    "nam",
    # cột kết quả/phân tích phim X-quang
    "chatluongphim",
    "ketqua",
    "matdotonthuong",
    "tonthuongkhac",
    "kichthuoctt",
}

pattern_drop = []
for col in df_raw.columns:
    c = str(col).strip()

    if re.match(r"^[BCDE]\s*\d+", c, flags=re.IGNORECASE):
        pattern_drop.append(col)
        continue

    m_f = re.match(r"^F\s*(\d+)", c, flags=re.IGNORECASE)
    if m_f:
        f_num = int(m_f.group(1))
        if 1 <= f_num <= 3 or 6 <= f_num <= 13:
            pattern_drop.append(col)
            continue

cols_to_drop = []
for c in df_raw.columns:
    if str(c).strip().lower() in explicit_drop:
        cols_to_drop.append(c)
cols_to_drop.extend(pattern_drop)

seen = set()
cols_to_drop = [c for c in cols_to_drop if not (c in seen or seen.add(c))]

before_cols = df_raw.shape[1]
df_pruned = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns], errors="ignore")
after_cols = df_pruned.shape[1]

print("\n✅ Pruned columns (chưa lưu Drive)")
print(f"- Cols before: {before_cols}")
print(f"- Cols dropped: {len(cols_to_drop)}")
print(f"- Cols after:  {after_cols}")

print("\nMột vài cột đã loại (tối đa 30):")
print(cols_to_drop[:30])

print("\nMột vài cột còn lại (tối đa 30):")
print(list(df_pruned.columns)[:30])

print("\nPreview 5 dòng đầu của df_pruned:")
display(df_pruned.head())

print("\nMissing summary (top 25 cột thiếu nhiều nhất) trước khi fill:")
_missing = df_pruned.isna().sum().sort_values(ascending=False)
_missing = _missing[_missing > 0]
display(_missing.head(25).to_frame(name="missing_count"))


In [ ]:
# === 2) Bổ sung data thiếu (text->"không", number->0) (chưa lưu Drive) ===

# Dùng df_pruned đã tạo ở Cell 1 (không tải thêm lần nữa)

# 1) Liệt kê cột bị thiếu
missing_counts = df_pruned.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]

print("\n### Missing columns (trước khi fill) ###")
print(f"Rows: {len(df_pruned):,} | Cols: {df_pruned.shape[1]:,} | Missing cols: {len(missing_counts):,}\n")

rows = []
for col, cnt in missing_counts.items():
    s = df_pruned[col]

    # nhận diện numeric vs text (cột numeric-like nhưng dtype object)
    if pd.api.types.is_numeric_dtype(s):
        kind = "numeric"
        convertible_ratio = 1.0
    else:
        s_non_missing = s.dropna()
        if len(s_non_missing) == 0:
            kind = "text"
            convertible_ratio = 0.0
        else:
            coerced = pd.to_numeric(s_non_missing, errors="coerce")
            convertible_ratio = float(coerced.notna().mean())
            kind = "numeric" if convertible_ratio >= 0.95 else "text"

    rows.append(
        {
            "column": str(col),
            "dtype": str(s.dtype),
            "kind": kind,
            "convertible_ratio": round(convertible_ratio, 4),
            "missing_count": int(cnt),
            "fill_value": 0 if kind == "numeric" else "không",
        }
    )

missing_report = pd.DataFrame(rows).sort_values("missing_count", ascending=False)
display(missing_report)

# 2) Fill missing theo rule
filled_df = df_pruned.copy()

num_cols = [r["column"] for r in rows if r["kind"] == "numeric"]
text_cols = [r["column"] for r in rows if r["kind"] == "text"]

for c in num_cols:
    filled_df[c] = pd.to_numeric(filled_df[c], errors="coerce").fillna(0)

if text_cols:
    filled_df[text_cols] = filled_df[text_cols].fillna("không")

# 3) Xử lý thêm các bất thường đặc thù trước khi mã hoá

# 3.1) Sửa giá trị bất thường ở F4gang (có 1 ô = 6.0) -> map về mode của cột
if "F4gang" in filled_df.columns:
    # Ép kiểu toàn bộ về string để dễ xử lý và loại bỏ khoảng trắng thừa
    filled_df["F4gang"] = filled_df["F4gang"].astype(str).str.strip()
    
    # Tạo mask tìm vị trí của giá trị lỗi '6.0' (hoặc '6')
    outlier_mask = filled_df["F4gang"].isin(["6.0", "6"])
    n_outliers = outlier_mask.sum()
    
    if n_outliers > 0:
        # Lấy tập dữ liệu hợp lệ (không chứa lỗi) để tìm mode thực sự
        valid_data = filled_df.loc[~outlier_mask, "F4gang"]
        
        if not valid_data.empty:
            true_mode = valid_data.mode().iloc[0]
            
            # Gán đè giá trị mode vào các ô bị lỗi
            filled_df.loc[outlier_mask, "F4gang"] = true_mode
            print(f"✅ Đã thay {n_outliers} giá trị '6.0' ở F4gang bằng mode = '{true_mode}'.")
        else:
            print("Cột F4gang không có dữ liệu hợp lệ để lấy mode.")
    else:
        print("Không tìm thấy giá trị '6.0' nào trong F4gang, không cần sửa.")
else:
    print("Không tìm thấy cột F4gang trong filled_df, bỏ qua bước sửa outlier.")

# 3.2) Tách riêng cột file_name thành mảng để ánh xạ sang ảnh X-quang
if "file_name" in filled_df.columns:
    file_names = filled_df["file_name"].astype(str).values
    print("\nĐã tách cột file_name ra mảng `file_names` (len =", len(file_names), ")")
    filled_df = filled_df.drop(columns=["file_name"])
    print("Cột `file_name` đã được loại khỏi bảng feature để dành riêng cho nhánh ảnh.")
else:
    file_names = None
    print("\nKhông tìm thấy cột file_name trong filled_df.")

# 4) In kiểm tra sau fill + xử lý đặc thù
remain_missing_cols = int((filled_df.isna().sum() > 0).sum())
remain_missing_cells = int(filled_df.isna().sum().sum())
print("\n✅ Filled + cleaned (chưa lưu Drive)")
print(f"- Remaining missing cols: {remain_missing_cols} | cells: {remain_missing_cells}")

print("\nPreview 5 dòng đầu sau fill + xử lý đặc thù:")
display(filled_df.head())


In [ ]:
# === 3) Feature Engineering + Encoding (theo spec) ===

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Làm việc trên bản copy để dễ rollback
# filled_df được tạo ở cell trước

df = filled_df.copy()

# ------------------------------
# Helpers
# ------------------------------

def _to_lower_str(s: pd.Series) -> pd.Series:
    # giữ NaN, còn lại chuyển về string lower + strip
    return s.astype("string").str.strip().str.lower()


def _binary_khong_or_nan_to_0_else_1(s: pd.Series) -> pd.Series:
    # 'khong' (không phân biệt hoa thường) hoặc NaN -> 0, còn lại -> 1
    ss = _to_lower_str(s)
    out = (~(ss.isna() | (ss == "khong"))).astype(int)
    # pandas StringDtype: ss.isna() đúng, còn giá trị thật NaN vẫn ok
    return out


def _safe_cols(cols):
    return [c for c in cols if c in df.columns]


# ------------------------------
# 1) Binary Encoding (0/1)
# ------------------------------

# gioitinh: Nam->0, Nu->1
if "gioitinh" in df.columns:
    g = _to_lower_str(df["gioitinh"])
    df["gioitinh"] = g.map({"nam": 0, "nu": 1}).fillna(0).astype(int)

# longnguc: Can doi->0, Khong can doi->1
if "longnguc" in df.columns:
    ln = _to_lower_str(df["longnguc"])
    df["longnguc"] = ln.map({"can doi": 0, "khong can doi": 1}).fillna(0).astype(int)

# khoangls, rungthan, go, riraopn: Binh thuong->0, còn lại->1
for c in _safe_cols(["khoangls", "rungthan", "go", "riraopn"]):
    ss = _to_lower_str(df[c])
    df[c] = (ss != "binh thuong").fillna(True).astype(int)  # NaN -> 1 theo rule "all other"; nhưng spec không nói NaN, giữ 1 để cảnh báo

# Danh sách cột: 'Khong'/NaN -> 0, còn lại -> 1
binary_khong_cols = [
    "cv5nam", "hutthuoc", "tiensuhh", "benhhh", "tsnoi", "tsngoai", "ho", "n1",
    "khacdom", "khotho", "daunguc", "daulan", "tcdau", "n2", "n3", "ytotang",
    "chaymui", "khan", "khokhe", "dhkhac", "metmoi", "sutcan", "vitridau",
    "ytodau", "rungt1", "rungt2", "rungt3", "rungt4", "rungt5", "rungt6",
    "rungg1", "rungg2", "rungg3", "rungg4", "rungg5", "rungg6", "goduc1",
    "goduc2", "goduc3", "goduc4", "goduc5", "goduc6", "ran", "ranam", "ranno",
    "ranrit", "ranngay", "vtam1", "vtam2", "vtam3", "vtam4", "vtam5", "vtam6",
    "vtno1", "vtno2", "vtno3", "vtno4", "vtno5", "vtno6", "vtrit1", "vtrit2",
    "vtrit3", "vtrit4", "vtrit5", "vtrit6", "vtngay1", "vtngay2", "vtngay3",
    "vtngay4", "vtngay5", "vtngay6", "A12", "bnn",
]

for c in _safe_cols(binary_khong_cols):
    df[c] = _binary_khong_or_nan_to_0_else_1(df[c])


# ------------------------------
# 2) Ordinal Encoding (custom mapping)
# ------------------------------

ordinal_maps = {
    "tdho": {"khong": 0, "ban ngay": 1, "ban dem": 2, "ca ngay va dem": 3},
    "tsho": {"khong": 0, "khac": 1, "tung con": 2, "lien tuc": 3},
    "loaidom": {"khong": 0, "0": 0, "dom nhay": 1, "dom mu": 2, "dom ba dau": 2, "dom mau": 3},
    "tdkhacdo": {"khong": 0, "ban ngay": 1, "ban dem": 2, "ca ngay va dem": 3},
    "mdkhotho": {"khong": 0, "khi gang suc": 1, "khi lam viec nhe": 2, "tung con": 3, "thuong xuyen": 4},
    "tdkhotho": {"khong": 0, "ban ngay": 1, "ban dem": 2, "ca ngay va dem": 3},
}

for col, mp in ordinal_maps.items():
    if col in df.columns:
        ss = _to_lower_str(df[col])
        df[col] = ss.map(mp).fillna(0).astype(int)

# Các cột bắt đầu bằng F4: mapping tần suất dùng BHLD
f4_map = {
    "khong su dung": 4,
    "hiem khi": 3,
    "thinh thoang": 2,
    "thuong xuyen": 1,
    "rat thuong xuyen": 0,
}

f4_cols = [c for c in df.columns if str(c).startswith("F4")]
for c in f4_cols:
    ss = _to_lower_str(df[c])
    df[c] = ss.map(f4_map).fillna(0).astype(int)


# ------------------------------
# 3) One-Hot Encoding
# ------------------------------

ohe_cols = _safe_cols(["A6", "A7", "A10", "A11"])
if ohe_cols:
    dummies = pd.get_dummies(df[ohe_cols], prefix=ohe_cols, drop_first=False)
    dummies = dummies.astype(int)
    df = pd.concat([df.drop(columns=ohe_cols), dummies], axis=1)


# ------------------------------
# 4) Numerical & Float Processing + StandardScaler
# ------------------------------

num_cols_all = [
    "namsinh", "tuoinghe", "nampx", "tgian1", "tgian2", "cao", "can", "hatd", "hatt",
    "mach", "theluc", "slthuoc", "tgdau", "socansut", "tgsut", "A9a", "A9b",
    "fvclt", "fvctt", "fev1lt", "fev1tt", "fvc", "fev1", "gaenler",
]

num_cols = _safe_cols(num_cols_all)

# Chuẩn hoá dấu phẩy -> dấu chấm, rồi ép float
for c in num_cols:
    if df[c].dtype == object or str(df[c].dtype).startswith("string"):
        df[c] = (
            df[c]
            .astype("string")
            .str.replace(",", ".", regex=False)
            .str.strip()
        )
    df[c] = pd.to_numeric(df[c], errors="coerce")

if num_cols:
    # Impute mean trước khi scale
    imputer = SimpleImputer(strategy="mean")
    scaler = StandardScaler()

    X_num = imputer.fit_transform(df[num_cols])
    X_scaled = scaler.fit_transform(X_num)

    df[num_cols] = X_scaled
    df[num_cols] = df[num_cols].fillna(0)


# ------------------------------
# 5) Skip embedding columns as requested
# ------------------------------
# ['cviec', 'cviec1', 'cviec2', 'pxuong'] giữ nguyên để sau đưa vào Embedding layer


print("\n✅ Encoding done")
print("Shape:", df.shape)
display(df.head())


In [ ]:
# === Text Normalization + Phân nhóm 10 cấp độ -> ID số nguyên (0-9) trên df ===

import re
import unicodedata

import pandas as pd

try:
    from unidecode import unidecode as _unidecode
except ImportError:
    _unidecode = None

JOB_COLS = ["cviec", "pxuong", "cviec1", "cviec2"]
LV9_ID = 8  # lv9: Không đi làm / không xác định

level_to_id = {f"lv{i}": i - 1 for i in range(1, 10)}  # lv1->0 ... lv9->8
level_to_id["lv10"] = 9


def _remove_vietnamese_accents(text: str) -> str:
    if _unidecode is not None:
        return _unidecode(text)
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def clean_job_text(text) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    s = str(text).strip().lower()
    if s in {"", "nan", "none", "<na>"}:
        return ""

    s = re.sub(r"\s+", " ", s)
    s = s.replace("đ", "d").replace("Đ", "d")
    s = _remove_vietnamese_accents(s)
    s = re.sub(r"\bsau chua\b", "sua chua", s)
    s = s.replace("x-ray", "x ray").replace("x ray", "x ray")
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _has_phrase(text: str, phrase: str) -> bool:
    if not phrase:
        return False
    if " " in phrase:
        return phrase in text
    return re.search(rf"\b{re.escape(phrase)}\b", text) is not None


def _match_keywords(text: str, keywords, short_boundary=None) -> bool:
    short_boundary = short_boundary or set()
    for kw in sorted(keywords, key=len, reverse=True):
        if kw in short_boundary:
            if re.search(rf"\b{re.escape(kw)}\b", text):
                return True
        elif _has_phrase(text, kw):
            return True
    return False


def classify_job_10_levels(clean_text: str) -> str:
    t = clean_text or ""

    if t == "" or t in {"khong", "nan", "o nha", "tu do"}:
        return "lv9"

    lv1_keywords = [
        "luyen", "duc", "nau", "nan", "no min", "ren", "dot", "chung cat",
        "khai thac", "cat", "nung",
        "coc", "thep", "gang", "silic", "da", "than", "lo", "mo", "kim loai",
        "xi mang", "clinker", "nghien", "amiang",
    ]
    if _match_keywords(t, lv1_keywords):
        return "lv1"

    lv2_keywords = [
        "gia cong", "xay", "boc", "boc vac", "khoan", "mai", "pha", "trang men",
        "trang tri", "tron", "cong truong", "phu ho", "xay dung", "dap",
        "gach", "gach men", "ceramic", "tho",
    ]
    if _match_keywords(t, lv2_keywords):
        return "lv2"

    lv3_keywords = [
        "phoi lieu", "tao hinh", "sua", "sua chua", "bao tri", "bao duong",
        "dien", "han", "ren", "co khi", "phay",
    ]
    if _match_keywords(t, lv3_keywords):
        return "lv3"

    lv4_keywords = [
        "can", "dong", "det", "phan loai", "xuat vo", "soi",
    ]
    if _match_keywords(t, lv4_keywords):
        return "lv4"
    if "bao ve" not in t and _has_phrase(t, "bao"):
        return "lv4"

    lv5_short = {"cn", "lam", "px", "vs", "con", "sx"}
    lv5_keywords = [
        "cong nhan", "lao dong", "san xuat", "xuat", "nguyen lieu", "nhap lieu",
        "nap lieu", "san pham", "say", "tron", "ve sinh", "xa", "truc", "bao ve",
        "cong ty", "cty", "kcn", "cong nghiep", "lien doanh", "nha may",
        "xi nghiep", "lap rap", "day chuyen",
        *lv5_short,
    ]
    if _match_keywords(t, lv5_keywords, short_boundary=lv5_short):
        return "lv5"

    lv6_short = {"vh", "dieu", "in", "ky"}
    lv6_keywords = [
        "chay", "dieu khien", "dung", "ky thuat", "lai", "may", "xe", "truc",
        "xu ly", "van chuyen", "van tai", *lv6_short,
    ]
    if _match_keywords(t, lv6_keywords, short_boundary=lv6_short):
        return "lv6"

    lv7_short = {"xn"}
    lv7_keywords = [
        "thi nghiem", "xet nghiem", "dem", "kiem", "ktra", "lay mau", "lab",
        "qc", "kcs", "x ray", *lv7_short,
    ]
    if _match_keywords(t, lv7_keywords, short_boundary=lv7_short):
        return "lv7"

    lv8_keywords = [
        "giam sat", "ke hoach", "kinh doanh", "kho vat tu", "quan ly", "ql", "qly",
        "giam doc", "nhan vien", "nv", "pho", "phong", "quan", "quat", "quy cach",
        "so lieu", "truong", "chi huy", "giay to", "thong", "cung cap", "ban",
        "buon ban", "chup anh", "chuyen vien", "di hoc", "bo doi", "quan su",
        "du lich", "ks", "nuoi", "nong nghiep", "gia suc", "gia cam", "noi",
        "quang cao", "sales", "shop", "giay", "quan ao", "thuc pham", "tiep thi",
        "bep", "nau an", "tap vu", "y te", "thiet ke", "ke toan",
    ]
    if _match_keywords(t, lv8_keywords, short_boundary={"ql", "nv", "ks"}):
        return "lv8"

    return "lv10"


# --- Xác nhận cột & xây raw_to_id_dict từ giá trị duy nhất ---
missing_cols = [c for c in JOB_COLS if c not in df.columns]
if missing_cols:
    raise KeyError(f"Thiếu cột trong df: {missing_cols}")

unique_values = pd.unique(pd.concat([df[c] for c in JOB_COLS], ignore_index=True))

raw_to_id_dict = {
    raw: level_to_id[classify_job_10_levels(clean_job_text(raw))]
    for raw in unique_values
}

# --- Ánh xạ trực tiếp vào df ---
for col in JOB_COLS:
    df[col] = df[col].map(raw_to_id_dict).fillna(LV9_ID).astype(int)

# --- Kiểm tra ---
print("✅ Đã chuyển 4 cột nghề nghiệp sang ID số nguyên (0-9) trên df")
print(f"- Số giá trị gốc duy nhất đã map: {len(raw_to_id_dict)}")
print(df[JOB_COLS].head())
print("\ndf['cviec'].value_counts():")
print(df["cviec"].value_counts())

In [ ]:
# === Tách nhánh dữ liệu đa phương thức (Job / LSTM 3D / Numerical) ===

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

JOB_COLS = ["cviec", "pxuong", "cviec1", "cviec2"]
TARGET_COL = "benhhh"
SEQ_PREFIXES = ["rungt", "rungg", "goduc", "vtam", "vtno", "vtrit", "vtngay"]
TEST_SIZE = 0.3
RANDOM_STATE = 42

# --- 1) Chuỗi lâm sàng không gian (LSTM Input) ---
seq_features = []
seq_cols_flat = []

for i in range(1, 7):
    zone_cols = [f"{p}{i}" for p in SEQ_PREFIXES if f"{p}{i}" in df.columns]
    if not zone_cols:
        raise ValueError(f"Không tìm thấy cột chuỗi cho vùng i={i}")
    seq_cols_flat.extend(zone_cols)
    seq_features.append(df[zone_cols].values.astype(np.float32))

X_seq_all = np.stack(seq_features, axis=1)

n_zones = X_seq_all.shape[1]
n_features_per_zone = X_seq_all.shape[2]
print(f"✅ LSTM sequence: {len(seq_cols_flat)} cột -> X_seq_all shape {X_seq_all.shape}")
print(f"   (N bệnh nhân, {n_zones} vùng, {n_features_per_zone} đặc trưng/vùng)")

# --- 2) Cột số (đã scale sẵn — KHÔNG re-scale) ---
exclude_cols = set(seq_cols_flat) | set(JOB_COLS) | {TARGET_COL,"tiensuhh"}

num_cols = [
    c
    for c in df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])
]

X_num_all = df[num_cols].to_numpy(dtype=np.float32)
imputer = SimpleImputer(strategy="mean")
X_num_all = imputer.fit_transform(X_num_all)

print(f"✅ Numerical: {len(num_cols)} cột (loại trừ target/job/seq)")

# --- 3) Target, job, train/val split ---
if TARGET_COL not in df.columns:
    raise KeyError(f"Thiếu cột target: {TARGET_COL}")

missing_job = [c for c in JOB_COLS if c not in df.columns]
if missing_job:
    raise KeyError(f"Thiếu cột nghề nghiệp: {missing_job}")

y = df[TARGET_COL].astype(int).to_numpy()
unique_y = set(np.unique(y))
if not unique_y.issubset({0, 1}):
    raise ValueError(f"{TARGET_COL} phải nhị phân 0/1, nhận được: {unique_y}")

X_job_all = df[JOB_COLS].to_numpy(dtype=np.int32)

idx = np.arange(len(df))
idx_train, idx_val = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_seq = X_seq_all[idx_train]
X_val_seq = X_seq_all[idx_val]
X_train_num = X_num_all[idx_train]
X_val_num = X_num_all[idx_val]
X_train_job = X_job_all[idx_train]
X_val_job = X_job_all[idx_val]
y_train = y[idx_train]
y_val = y[idx_val]

classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, weights)}

# --- 4) In shape & kiểm tra ---
print("\n" + "=" * 60)
print("SHAPES — toàn bộ mảng đầu ra")
print("=" * 60)
print(f"X_seq_all:      {X_seq_all.shape}")
print(f"X_train_seq:    {X_train_seq.shape}")
print(f"X_val_seq:      {X_val_seq.shape}")
print(f"X_num_all:      {X_num_all.shape}")
print(f"X_train_num:    {X_train_num.shape}")
print(f"X_val_num:      {X_val_num.shape}")
print(f"X_train_job:    {X_train_job.shape}")
print(f"X_val_job:      {X_val_job.shape}")
print(f"y:              {y.shape}")
print(f"y_train:        {y_train.shape}")
print(f"y_val:          {y_val.shape}")
print(f"class_weights_dict: {class_weights_dict}")

print("\nPhân phối nhãn (0=Khỏe, 1=Bệnh):")
print("  Toàn tập:", dict(zip(*np.unique(y, return_counts=True))))
print("  Train:  ", dict(zip(*np.unique(y_train, return_counts=True))))
print("  Val:    ", dict(zip(*np.unique(y_val, return_counts=True))))
print(f"\nTrain/Val: {len(idx_train)} / {len(idx_val)} ({100*(1-TEST_SIZE):.0f}% / {100*TEST_SIZE:.0f}%)")

In [ ]:
# === Multi-Input Keras Model (Functional API) ===

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam

# Shape động từ mảng đã chia (Cell trước)
n_seq_features = X_train_seq.shape[2]
n_num_features = X_train_num.shape[1]
n_job_slots = X_train_job.shape[1]
assert n_job_slots == 4, f"Kỳ vọng 4 cột nghề, nhận {n_job_slots}"

print(f"Input shapes: job=({n_job_slots},), seq=(6, {n_seq_features}), num=({n_num_features},)")

# --- Nhánh 1: Categorical Embedding (Nghề nghiệp) ---
job_input = layers.Input(shape=(n_job_slots,), name="job_input")
x_job = layers.Embedding(input_dim=10, output_dim=8, name="job_embedding")(job_input)
x_job = layers.GlobalAveragePooling1D(name="job_pool")(x_job)
x_job = layers.Dense(16, activation="relu", name="job_dense")(x_job)

# --- Nhánh 2: LSTM (Chuỗi lâm sàng) ---
seq_input = layers.Input(shape=(6, n_seq_features), name="seq_input")
x_seq = layers.LSTM(32, return_sequences=False, name="lstm_clinical")(seq_input)
x_seq = layers.Dense(16, activation="relu", name="seq_dense")(x_seq)

# --- Nhánh 3: Dense (Chỉ số số đã scale) ---
num_input = layers.Input(shape=(n_num_features,), name="num_input")
x_num = layers.Dense(16, activation="relu", name="num_dense")(num_input)

# --- Fusion & Classification Head ---
merged = layers.Concatenate(name="fusion_concat")([x_job, x_seq, x_num])
x = layers.Dense(32, activation="relu", name="fusion_dense")(merged)
x = layers.Dropout(0.3, name="fusion_dropout")(x)
output = layers.Dense(1, activation="sigmoid", name="output")(x)

model = Model(
    inputs=[job_input, seq_input, num_input],
    outputs=output,
    name="multimodal_bnn_classifier",
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model.summary()

In [ ]:
# === Huấn luyện & Đánh giá mô hình (Validation 30%) ===

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# --- 1) Training ---
history = model.fit(
    x=[X_train_job, X_train_seq, X_train_num],
    y=y_train,
    validation_data=([X_val_job, X_val_seq, X_val_num], y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weights_dict,
    verbose=1,
)

h = history.history
last = len(h["loss"]) - 1
print(
    f"\nEpoch cuối ({last + 1}): "
    f"loss={h['loss'][last]:.4f}, val_loss={h['val_loss'][last]:.4f}, "
    f"auc={h['auc'][last]:.4f}, val_auc={h['val_auc'][last]:.4f}"
)

# --- 2) Learning curves (lưu PNG) ---
epochs_range = range(1, len(h["loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, h["loss"], label="Train")
plt.plot(epochs_range, h["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve - Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curve_loss.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, h["auc"], label="Train")
plt.plot(epochs_range, h["val_auc"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Learning Curve - AUC")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curve_auc.png", dpi=150)
plt.show()

print("Đã lưu: learning_curve_loss.png, learning_curve_auc.png")

# --- 3) Đánh giá Validation ---
y_val_prob = model.predict(
    [X_val_job, X_val_seq, X_val_num],
    batch_size=64,
    verbose=0,
).ravel()

y_val_pred = (y_val_prob >= 0.5).astype(int)

cm = confusion_matrix(y_val, y_val_pred)
print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm)

target_names = ["Khỏe (0)", "Bệnh (1)"]
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, target_names=target_names, digits=4))